# RAPID Predictive Classifiers — Tutorial Notebook

This notebook walks you through the **predictive modelling pipeline** of RAPID: Logistic Regression (L2), Decision Tree, Random Forest, SVM and XGBoost, all built on top of the same `RAPID_MLBaseClassifier` template.

By the end of this tutorial you will know how to:
- Instantiate any of the five predictive classifiers through the **RAPID Pipeline Factory**
- Configure **temporal validation** (train on earlier years, test on later years) instead of a random split
- Use **cyclical (sine/cosine) encoding** for seasonal variables such as month or epidemiological week
- Read **extended discrimination metrics** (AUC-ROC, AUC-PR, NPV, Specificity) at a fold-safe decision threshold
- Read the **collinearity diagnostic report** (correlation matrix + VIF)
- Generate **SHAP** interpretability plots for the tree-based models (Random Forest, XGBoost)
- Compare all five models side by side

> **This notebook is dataset-independent.** It generates a synthetic dataset in Section 2 so it runs end-to-end with no external files. To use it with your own data, replace the `df` DataFrame and update the column names passed to `factory.create()` — every other cell stays the same. See Section 9 for a checklist.

## 1. Setup and Installation

Install RAPID from the repository root (editable install is recommended during development):

```bash
pip install -e .
```

In [ ]:
from isaric.modeling.pipeline_factory import RAPID_PipelineFactory
import pandas as pd
import numpy as np

## 2. A Synthetic Dataset (swap this for your own data)

The cell below builds a self-contained synthetic dataset that mimics a typical arbovirus surveillance record: a rare binary outcome (`hospitalized`), a handful of binary/continuous predictors with some missing values (to exercise MICE imputation), a `year` column for the temporal split, a date column and an epidemiological-week column for the cyclical encoding.

In [ ]:
np.random.seed(42)
n = 4000

event_date = pd.to_datetime("2018-01-01") + pd.to_timedelta(
    np.random.randint(0, 365 * 6, size=n), unit="D"
)

df = pd.DataFrame({
    "year": event_date.year,
    "event_date": event_date,
    "epi_week": [f"{d.year}{d.isocalendar()[1]:02d}" for d in event_date],
    "age": np.random.randint(18, 90, size=n),
    "sex_male": np.random.choice([0, 1], size=n),
    "fever": np.random.binomial(1, 0.75, size=n).astype(float),
    "myalgia": np.random.binomial(1, 0.6, size=n).astype(float),
    "comorbidity_score": np.random.poisson(lam=1.5, size=n).astype(float),
    "warning_sign": np.random.binomial(1, 0.15, size=n).astype(float),
})

# a handful of missing values, to exercise the MICE imputation step
for col in ["age", "fever", "comorbidity_score"]:
    missing_mask = np.random.random(n) < 0.05
    df.loc[missing_mask, col] = np.nan

# rare, seasonally-modulated binary outcome (hospitalization-like)
seasonal = 0.02 * np.sin(2 * np.pi * event_date.month / 12)
log_odds = (
    -4.2
    + 0.02 * df["age"].fillna(df["age"].median())
    + 1.1 * df["warning_sign"]
    + 0.5 * df["comorbidity_score"].fillna(0)
    + seasonal
)
prob = 1 / (1 + np.exp(-log_odds))
df["hospitalized"] = np.random.binomial(1, np.clip(prob, 0, 1))

print("Outcome prevalence:", df["hospitalized"].mean())
df["year"].value_counts().sort_index()

**Important**: `dependent_var` must already be coded as `0`/`1` — categorical outcomes (e.g. `"Yes"`/`"No"`) must be recoded before calling `factory.create()`. The same applies to categorical predictors: `independent_vars` should be numeric/binary (e.g. `sex_male` as a 0/1 dummy), not raw category strings.

## 3. The Pipeline Factory

All RAPID pipelines — inferential regressions and the predictive classifiers alike — are created through the same **`RAPID_PipelineFactory`**.

In [ ]:
factory = RAPID_PipelineFactory()
print(factory.available())

The five predictive classifiers added in this MVP are registered as `"logistic_l2"`, `"decision_tree"`, `"random_forest"`, `"svm"` and `"xgboost"`.

## 4. Creating and Fitting a Model (XGBoost example)

Key parameters, beyond the usual `data`/`dependent_var`/`independent_vars`:

- `year_column`, `train_end_year`, `test_start_year` — define the **temporal split**: rows with `year_column <= train_end_year` go to training, rows with `year_column >= test_start_year` go to testing. This replaces a random train/test split to avoid leaking future information into the model.
- `date_column` / `epiweek_column` (optional) — enable **cyclical (sine/cosine) encoding** of month and epidemiological week, useful for seasonal diseases like dengue.
- `imputation_strategy` — `"mice"` (default, `IterativeImputer`) or `"median"`/`"mode"` (sensitivity analysis).
- `imbalance_strategy` — `None` (default: relies on `class_weight='balanced'` / `scale_pos_weight`), or `"smote"`/`"undersample"` (sensitivity analysis, applied only inside the training folds).
- `cv_splits` / `cv_repeats` — repeated stratified k-fold used for hyperparameter tuning **inside the training block only**.
- `threshold_method` — `"f1"` (default) or `"youden"`; the decision threshold is selected from out-of-fold training predictions, never from the test set.

In [ ]:
independent_vars = ["age", "sex_male", "fever", "myalgia", "comorbidity_score"]

model = factory.create(
    "xgboost",
    data=df,
    dependent_var="hospitalized",
    independent_vars=independent_vars,
    year_column="year",
    train_end_year=2022,
    test_start_year=2023,
    date_column="event_date",
    epiweek_column="epi_week",
    cv_splits=3,
    cv_repeats=2,
    n_iter=5,
    n_jobs=1,
)
model.fit()

print("best_params_:", model.best_params_)
print("threshold_:", model.threshold_)

## 5. Performance Metrics

`summary()` prints the extended discrimination metrics (Accuracy, Precision, Recall, F1, Specificity, NPV, AUC-ROC, AUC-PR, Brier Score) evaluated on the temporal test block, at the fold-safe threshold selected during `fit()`.

In [ ]:
model.summary(performance="all", collinearity=None, plots=None)

## 6. Collinearity Diagnostics

The collinearity report combines a Pearson correlation matrix with Variance Inflation Factors (VIF). It is **diagnostic only** — no variable is removed automatically, the decision is left to the analyst.

In [ ]:
model.summary(performance=None, collinearity="all", plots=None)

## 7. SHAP Interpretability (tree-based models only)

SHAP summary and beeswarm plots are available for the tree-based models — **Random Forest and XGBoost** — via `TreeSHAPMixin`. They are saved as PNG artifacts in the current working directory.

In [ ]:
model.summary(performance=None, collinearity=None, plots=["shap_summary", "shap_beeswarm"])

Calling `plots=["shap_summary", "shap_beeswarm"]` on a model that does not implement `TreeSHAPMixin` (Logistic Regression, Decision Tree, SVM) raises a `NotImplementedError` — this is expected: SHAP is scoped to Random Forest and XGBoost in this MVP.

## 8. Comparing All Five Predictive Models

In [ ]:
comparison_rows = []
for name in ["logistic_l2", "decision_tree", "random_forest", "svm", "xgboost"]:
    m = factory.create(
        name,
        data=df,
        dependent_var="hospitalized",
        independent_vars=independent_vars,
        year_column="year",
        train_end_year=2022,
        test_start_year=2023,
        date_column="event_date",
        epiweek_column="epi_week",
        cv_splits=3,
        cv_repeats=1,
        n_iter=3,
        n_jobs=1,
    )
    m.fit()
    metrics = m.performance_metrics_
    comparison_rows.append({
        "model": name,
        "auc_roc": metrics["auc_roc"],
        "auc_pr": metrics["auc_pr"],
        "f1": metrics["f1"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "specificity": metrics["specificity"],
        "npv": metrics["npv"],
        "brier_score": metrics["brier_score"],
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("auc_roc", ascending=False)
comparison_df

## 9. Using Your Own Dataset — Checklist

To point this notebook at a real dataset instead of the synthetic one from Section 2:

1. Replace `df` with your own `pandas.DataFrame`.
2. Make sure `dependent_var` is coded as `0`/`1` (recode string/categorical outcomes beforehand).
3. Make sure every column in `independent_vars` is numeric/binary (one-hot encode categorical predictors beforehand, e.g. sex dummies).
4. Set `year_column` to a real year column in your data (e.g. `ano_sin_pri`), and choose `train_end_year`/`test_start_year` so both the training and test blocks have enough rows.
5. `date_column`/`epiweek_column` are optional — omit them (leave as `None`) if you don't have a usable date or epidemiological-week column.
6. Everything else — the factory call, `fit()`, `summary()`, the collinearity report and the SHAP plots — stays exactly the same.

## 10. Sensitivity Analyses

Two parameters are designed as one-line sensitivity toggles, matching the MVP scope:

- `imputation_strategy="median"` or `"mode"` instead of the default `"mice"`.
- `imbalance_strategy="smote"` or `"undersample"` instead of the default `None` (class-weight-only). Both are applied exclusively inside the training folds — never on the full dataset — to avoid leaking resampled information into the test block.

In [ ]:
model_sensitivity = factory.create(
    "random_forest",
    data=df,
    dependent_var="hospitalized",
    independent_vars=independent_vars,
    year_column="year",
    train_end_year=2022,
    test_start_year=2023,
    imputation_strategy="median",
    imbalance_strategy="smote",
    cv_splits=3,
    cv_repeats=1,
    n_iter=3,
    n_jobs=1,
)
model_sensitivity.fit()
model_sensitivity.summary(performance="all", collinearity=None, plots=None)